# Risk Theme Classifier: Two-SVM Ensemble

Labels each review with `dependence`, `attachment`, `isolation_displacement`, `crisis_response_frustration` and/or `other_none`.

The model blends two linear SVMs, one on sentence embeddings (meaning) and one on TF-IDF (exact words). They make different mistakes on individual reviews, so averaging their scores beats either one alone (see `svm_experiment_log.md`).

- **Train:** run sections 1–3.
- **Predict with the saved model:** run sections 1, 2 and 4.

## 1. Setup

In [1]:
from itertools import product

import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC

SEED = 42
MODEL_PATH = "../models/two_svm_ensemble.joblib"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LABEL_NAMES = ["dependence", "attachment", "isolation_displacement", "crisis_response_frustration", "other_none"]
OTHER_INDEX = LABEL_NAMES.index("other_none")

encoder = SentenceTransformer(EMBEDDING_MODEL)


def encode(texts):
    return encoder.encode(list(texts), batch_size=32, show_progress_bar=False, normalize_embeddings=True)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1101 16:54:59.389000 58313 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12557.80it/s]

## 2. The ensemble
Each SVM's scores are divided by their cross-validation spread so the two are on the same scale, then averaged with a tuned weight. A label is predicted when the blended score passes a tuned threshold. The cleanup rule then removes `other_none` when a risk theme is predicted, and assigns it when nothing is. Settings are tuned by 5-fold cross-validation (seeds 0, 1, 2).

In [2]:
C_GRID = [0.03, 0.1, 0.3, 1]
EMBEDDING_WEIGHTS = [0, 0.25, 0.5, 0.75, 1]
SELECTION_SHIFTS = [0.2, 0.1, 0, -0.1, -0.2, -0.3, -0.4]
THRESHOLD_SHIFTS = [round(value, 1) for value in np.arange(-0.4, 1.25, 0.1)]


def fit_svms(features, Y, C_value):
    return [
        LinearSVC(C=C_value, class_weight="balanced", random_state=SEED).fit(features, Y[:, label_index])
        for label_index in range(Y.shape[1])
    ]


def svm_scores(models, features):
    return np.column_stack([model.decision_function(features) for model in models])


def clean_predictions(predictions):
    cleaned = predictions.copy()
    cleaned[cleaned[:, :OTHER_INDEX].any(axis=1), OTHER_INDEX] = 0
    cleaned[~cleaned.any(axis=1), OTHER_INDEX] = 1
    return cleaned


class TwoSVMEnsemble:
    def __init__(self, embedding_C, tfidf_C, embedding_weight, shift, embedding_scale, tfidf_scale):
        self.embedding_C, self.tfidf_C = embedding_C, tfidf_C
        self.embedding_weight, self.shift = embedding_weight, shift
        self.embedding_scale, self.tfidf_scale = embedding_scale, tfidf_scale

    @staticmethod
    def _member_scores(member, train_texts, train_embeddings, train_Y, score_texts, score_embeddings, C_value):
        if member == "embedding":
            return svm_scores(fit_svms(train_embeddings, train_Y, C_value), score_embeddings)
        vectorizer = TfidfVectorizer(sublinear_tf=True, strip_accents="unicode")
        models = fit_svms(vectorizer.fit_transform(train_texts), train_Y, C_value)
        return svm_scores(models, vectorizer.transform(score_texts))

    @classmethod
    def tune(cls, texts, Y, embeddings):
        """Choose each SVM's C, then the blend weight and threshold, by repeated 5-fold CV."""
        frequencies = Y.sum(axis=0)
        strata = np.array([min(np.flatnonzero(row), key=lambda j: frequencies[j]) for row in Y])
        folds_by_seed = [
            list(StratifiedKFold(n_splits=5, shuffle=True, random_state=cv_seed).split(texts, strata))
            for cv_seed in (0, 1, 2)
        ]
        scaled, scales = {}, {}
        for member, C_value in product(["embedding", "tfidf"], C_GRID):
            scores_by_seed = []
            for folds in folds_by_seed:
                scores = np.zeros(Y.shape)
                for train, validation in folds:
                    scores[validation] = cls._member_scores(
                        member, texts[train], embeddings[train], Y[train],
                        texts[validation], embeddings[validation], C_value,
                    )
                scores_by_seed.append(scores)
            scales[member, C_value] = np.mean([scores.std(axis=0) for scores in scores_by_seed], axis=0)
            scaled[member, C_value] = [scores / scales[member, C_value] for scores in scores_by_seed]

        def cv_macro_f1(predict_seed):
            return np.mean([
                f1_score(Y, predict_seed(seed), average="macro", zero_division=0) for seed in range(len(folds_by_seed))
            ])

        best_C = {
            member: max(C_GRID, key=lambda C_value: max(
                cv_macro_f1(lambda seed: (scaled[member, C_value][seed] > shift).astype(int))
                for shift in SELECTION_SHIFTS
            ))
            for member in ["embedding", "tfidf"]
        }
        embedding_scores, tfidf_scores = scaled["embedding", best_C["embedding"]], scaled["tfidf", best_C["tfidf"]]
        weight, shift = max(
            ((weight, shift) for weight in EMBEDDING_WEIGHTS for shift in THRESHOLD_SHIFTS),
            key=lambda option: cv_macro_f1(lambda seed: clean_predictions((
                option[0] * embedding_scores[seed] + (1 - option[0]) * tfidf_scores[seed] > option[1]
            ).astype(int))),
        )
        return cls(best_C["embedding"], best_C["tfidf"], weight, shift,
                   scales["embedding", best_C["embedding"]], scales["tfidf", best_C["tfidf"]])

    def fit(self, texts, Y, embeddings):
        self.embedding_models = fit_svms(embeddings, Y, self.embedding_C)
        self.vectorizer = TfidfVectorizer(sublinear_tf=True, strip_accents="unicode")
        self.tfidf_models = fit_svms(self.vectorizer.fit_transform(texts), Y, self.tfidf_C)
        return self

    def predict(self, texts, embeddings=None):
        """0/1 label matrix (columns in LABEL_NAMES order)."""
        texts = np.asarray(texts, dtype=object)
        if len(texts) == 0:
            return np.zeros((0, len(LABEL_NAMES)), dtype=int)
        embeddings = encode(texts) if embeddings is None else embeddings
        embedding_scores = svm_scores(self.embedding_models, embeddings) / self.embedding_scale
        tfidf_scores = svm_scores(self.tfidf_models, self.vectorizer.transform(texts)) / self.tfidf_scale
        blended = self.embedding_weight * embedding_scores + (1 - self.embedding_weight) * tfidf_scores
        return clean_predictions((blended > self.shift).astype(int))

    def label_reviews(self, texts):
        """Reviews with Yes/blank label columns, in the training CSV's format."""
        predictions = self.predict(texts)
        frame = pd.DataFrame({"review": list(texts)})
        for label_index, name in enumerate(LABEL_NAMES):
            frame["label_" + name] = np.where(predictions[:, label_index] == 1, "Yes", "")
        return frame

    def save(self, path):
        joblib.dump(vars(self), path)

    @classmethod
    def load(cls, path):
        model = cls.__new__(cls)
        vars(model).update(joblib.load(path))
        return model

## 3. Train on all 600 reviews and save

In [3]:
df = pd.read_csv("annotated_training_test_set_600_samples.csv")
LABELS = ["label_" + name for name in LABEL_NAMES]
Y = df[LABELS].apply(lambda column: column.astype(str).str.strip().str.lower().eq("yes").astype(int)).values
texts = np.asarray(df["review"].fillna("").astype(str), dtype=object)
X = encode(texts)

model = TwoSVMEnsemble.tune(texts, Y, X).fit(texts, Y, X)
model.save(MODEL_PATH)
print(f"Trained on {len(df)} reviews: embedding C={model.embedding_C}, TF-IDF C={model.tfidf_C}, "
      f"embedding weight {model.embedding_weight}, threshold {model.shift:.1f}")

Trained on 600 reviews: embedding C=0.3, TF-IDF C=0.3, embedding weight 0.5, threshold -0.1


## 4. Load the saved model and predict new reviews

In [4]:
model = TwoSVMEnsemble.load(MODEL_PATH)

new_reviews = [
  # Example reviews to label
]
model.label_reviews(new_reviews)

,review,label_dependence,label_attachment,label_isolation_displacement,label_crisis_response_frustration,label_other_none
